# 02 — Canonical time-series EDA

**Purpose:** determine whether canonical telemetry is statistically suitable
for anomaly detection and record the evidence needed by feature engineering.

This notebook runs unchanged for Telecom and Petrobras 3W. It reads only
`SPEC-CORE`; evaluation truth is not an input.


## 1. Guardrails

The notebook has two scopes:

1. a streaming structural audit over all canonical telemetry;
2. statistical analysis inside the first, frozen fraction of each selected
   entity history.

The frozen interval is **label-blind**, not guaranteed normal. The notebook
does not remove outliers, create anomaly scores, tune thresholds, or fit a
production model.


In [ ]:
import hashlib
import json
import os
import shutil
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display
from scipy.stats import skew, spearmanr
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_SCHEMAS,
    CORE_VERSION,
    immutable_directory,
    read_json,
    sha256_file,
    write_json,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

figure_temp = tempfile.TemporaryDirectory()
FIGURE_CACHE = Path(figure_temp.name)


def save_and_show(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURE_CACHE / filename, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


### Choose the sector and analysis scope

Change `SECTOR` to run the same notebook on another canonical sector. The
entity limit bounds statistical work; the structural audit still scans every
telemetry partition.

Change `EDA_RUN_ID` before repeating a completed run because EDA outputs are
immutable.


In [ ]:
# Change only this value: "telecom" or "petrobras_3w"
SECTOR = "telecom"
SECTOR = os.getenv("EDA_SECTOR", SECTOR)

CORE_RUN_IDS = {
    "telecom": "telecom_canonical_v1",
    "petrobras_3w": "petrobras_3w_canonical_v1",
}
if SECTOR not in CORE_RUN_IDS:
    raise ValueError(f"Choose one of {list(CORE_RUN_IDS)}")

CORE_RUN_ID = os.getenv(
    "EDA_CORE_RUN_ID",
    CORE_RUN_IDS[SECTOR],
)
EDA_RUN_ID = os.getenv("EDA_RUN_ID", "eda_v1")

CALIBRATION_FRACTION = 0.40
ENTITY_LIMIT = 20
MAX_CALIBRATION_ROWS = 4_000_000
MAX_PLOT_POINTS = 5_000
MAX_DISTRIBUTION_POINTS = 20_000
MAX_CORRELATION_METRICS = 15
MAX_TIME_PLOT_METRICS = 8
MIN_STL_CYCLES = 6
SAVE_OUTPUTS = True

CORE_ROOT = (
    DRIVE_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}"
    / SECTOR / CORE_RUN_ID / "SPEC-CORE"
)
EDA_ROOT = (
    DRIVE_ROOT / "outputs" / "eda" / f"v{CORE_VERSION}"
    / SECTOR / EDA_RUN_ID
)

display(pd.Series({
    "sector": SECTOR,
    "spec_core": str(CORE_ROOT),
    "eda_output": str(EDA_ROOT),
    "calibration_fraction": CALIBRATION_FRACTION,
    "entity_limit": ENTITY_LIMIT,
}, name="value").to_frame())


## 2. Load the canonical contract safely

All reads go through the functions below. They reject any path outside the
selected `SPEC-CORE` directory and record every input file for the EDA
manifest.


In [ ]:
if not CORE_ROOT.is_dir():
    raise FileNotFoundError(CORE_ROOT)
if SAVE_OUTPUTS and EDA_ROOT.exists():
    raise FileExistsError(
        f"EDA output already exists. Change EDA_RUN_ID: {EDA_ROOT}"
    )

CORE_RESOLVED = CORE_ROOT.resolve()
READ_FILES = set()


def checked_core_path(relative_path):
    path = (CORE_ROOT / relative_path).resolve()
    if not path.is_relative_to(CORE_RESOLVED):
        raise PermissionError(f"Read outside SPEC-CORE: {path}")
    if not path.is_file():
        raise FileNotFoundError(path)
    READ_FILES.add(path)
    return path


def read_core_parquet(relative_path, columns=None):
    return pd.read_parquet(
        checked_core_path(relative_path),
        columns=columns,
    )


def read_telemetry_part(path, columns=None):
    path = Path(path).resolve()
    if not path.is_relative_to(CORE_RESOLVED):
        raise PermissionError(f"Read outside SPEC-CORE: {path}")
    READ_FILES.add(path)
    return pd.read_parquet(path, columns=columns)


manifest_path = checked_core_path("manifest.json")
core_manifest = read_json(manifest_path)
catalogue = read_core_parquet("metric_catalogue.parquet")
registry = read_core_parquet("entity_registry.parquet")
relations = read_core_parquet("entity_relations.parquet")
collection_gaps = read_core_parquet("collection_gaps.parquet")
telemetry_parts = sorted((CORE_ROOT / "telemetry").glob("part-*.parquet"))

assert telemetry_parts
assert core_manifest["contract_version"] == CORE_VERSION
assert list(catalogue.columns) == CORE_SCHEMAS["metric_catalogue"]
assert list(registry.columns) == CORE_SCHEMAS["entity_registry"]
assert list(relations.columns) == CORE_SCHEMAS["entity_relations"]
assert list(collection_gaps.columns) == CORE_SCHEMAS["collection_gaps"]


## 3. See the data and understand the fields

The metric catalogue is the canonical feature dictionary. A small telemetry
sample is also pivoted to a familiar wide form for inspection; the stored
canonical representation remains long.


In [ ]:
print("Canonical manifest")
display(pd.Series({
    "contract_version": core_manifest["contract_version"],
    "sector": core_manifest["sector"],
    "cadence_seconds": core_manifest["expected_cadence_seconds"],
    **core_manifest["row_counts"],
}, name="value").to_frame())

print("Metric catalogue")
display(catalogue)

print("Entities by type")
display(
    registry.groupby("entity_type", as_index=False)
    .agg(entities=("entity_id", "nunique"))
)

print("Relation types")
if relations.empty:
    print("No relations supplied by this sector pack.")
else:
    display(
        relations.groupby("relation_type", as_index=False)
        .size().rename(columns={"size": "relations"})
    )

sample = read_telemetry_part(telemetry_parts[0]).head(20)
print("Canonical long telemetry")
display(sample)

wide_sample = sample.pivot_table(
    index=["event_ts", "entity_id"],
    columns="metric_id",
    values="value",
    aggfunc="first",
).reset_index()
print("The same sample in wide form")
display(wide_sample)


## 4. Deterministic entity selection and full structural audit

Entities are chosen by a stable hash, not a changing random sample. The audit
below scans every telemetry partition to quantify quality by metric. It also
finds the observed time range for the selected entities.


In [ ]:
telemetry_entity_types = set(catalogue["entity_type"].astype(str))
candidate_entities = sorted(
    registry.loc[
        registry["entity_type"].astype(str).isin(telemetry_entity_types),
        "entity_id",
    ].astype(str).unique()
)
if not candidate_entities:
    raise ValueError("No telemetry entities found in entity_registry")


def stable_entity_key(entity_id):
    text = f"canonical-eda-v1|{SECTOR}|{entity_id}"
    return hashlib.sha256(text.encode()).hexdigest()


selected_entities = sorted(
    candidate_entities,
    key=stable_entity_key,
)[:ENTITY_LIMIT]
selected_set = set(selected_entities)

print(
    f"Selected {len(selected_entities):,} of "
    f"{len(candidate_entities):,} telemetry entities"
)
display(pd.DataFrame({"entity_id": selected_entities}))


In [ ]:
structural_rows = []
entity_bounds = []
duplicate_rows = 0
telemetry_columns = [
    "event_ts", "entity_id", "metric_id", "value", "quality_code"
]

for part in telemetry_parts:
    frame = read_telemetry_part(part, telemetry_columns)
    frame["event_ts"] = pd.to_datetime(frame["event_ts"], utc=True)
    frame["entity_id"] = frame["entity_id"].astype(str)
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")

    for metric_id, group in frame.groupby("metric_id", sort=False):
        structural_rows.append({
            "metric_id": metric_id,
            "rows": len(group),
            "valid": group["value"].notna().sum(),
            "invalid": group["quality_code"].eq("invalid").sum(),
            "clipped": group["quality_code"].eq("clipped").sum(),
            "zero": group["value"].eq(0).sum(),
        })

    duplicate_rows += int(frame.duplicated(
        ["event_ts", "entity_id", "metric_id"]
    ).sum())

    selected = frame.loc[frame["entity_id"].isin(selected_set)]
    if not selected.empty:
        bounds = (
            selected.groupby("entity_id")["event_ts"]
            .agg(observed_start="min", observed_end="max")
            .reset_index()
        )
        entity_bounds.append(bounds)

structural_profile = (
    pd.DataFrame(structural_rows)
    .groupby("metric_id", as_index=False)
    .sum(numeric_only=True)
)
for column in ("valid", "invalid", "clipped", "zero"):
    structural_profile[f"{column}_rate"] = (
        structural_profile[column] / structural_profile["rows"]
    )

bounds = (
    pd.concat(entity_bounds)
    .groupby("entity_id", as_index=False)
    .agg(observed_start=("observed_start", "min"),
         observed_end=("observed_end", "max"))
)
assert set(selected_entities) <= set(bounds["entity_id"])

print(f"Duplicate canonical keys within partitions: {duplicate_rows:,}")
display(structural_profile)


### Collection gaps and quality overview

Collection gaps are expected observations that were absent. Invalid values are
different: their rows exist, but their values are unusable.


In [ ]:
gap_summary = collection_gaps.copy()
if gap_summary.empty:
    print("No collection gaps were recorded.")
else:
    gap_summary["gap_start"] = pd.to_datetime(
        gap_summary["gap_start"], utc=True
    )
    gap_summary["gap_end"] = pd.to_datetime(
        gap_summary["gap_end"], utc=True
    )
    gap_summary["duration_seconds"] = (
        gap_summary["gap_end"] - gap_summary["gap_start"]
    ).dt.total_seconds()
    display(gap_summary["duration_seconds"].describe().to_frame())

quality_plot = structural_profile.set_index("metric_id")[
    ["invalid_rate", "clipped_rate"]
]
fig, ax = plt.subplots(figsize=(11, 5))
quality_plot.plot.bar(stacked=True, ax=ax, color=["#d95f02", "#7570b3"])
ax.set_title(f"{SECTOR}: invalid and clipped observation rates")
ax.set_xlabel("")
ax.set_ylabel("Fraction of rows")
ax.tick_params(axis="x", rotation=65)
save_and_show(fig, "quality_rates.png")


## 5. Freeze label-blind calibration windows

Each selected entity contributes only the first 40% of its observed time span
to statistical analysis. These windows are saved before distributions or
temporal patterns are assessed.


In [ ]:
analysis_windows = bounds.copy()
duration = (
    analysis_windows["observed_end"]
    - analysis_windows["observed_start"]
)
analysis_windows["calibration_end"] = (
    analysis_windows["observed_start"]
    + duration * CALIBRATION_FRACTION
)
analysis_windows["calibration_fraction"] = CALIBRATION_FRACTION
display(analysis_windows)

calibration_end = analysis_windows.set_index(
    "entity_id"
)["calibration_end"]
calibration_parts = []

for part in telemetry_parts:
    frame = read_telemetry_part(part, telemetry_columns)
    frame["event_ts"] = pd.to_datetime(frame["event_ts"], utc=True)
    frame["entity_id"] = frame["entity_id"].astype(str)
    frame = frame.loc[frame["entity_id"].isin(selected_set)].copy()
    if frame.empty:
        continue
    end = frame["entity_id"].map(calibration_end)
    frame = frame.loc[frame["event_ts"].le(end)]
    if not frame.empty:
        calibration_parts.append(frame)

calibration = pd.concat(calibration_parts, ignore_index=True)
calibration["value"] = pd.to_numeric(
    calibration["value"], errors="coerce"
)
calibration["entity_id"] = calibration["entity_id"].astype("category")
calibration["metric_id"] = calibration["metric_id"].astype("category")
calibration["quality_code"] = calibration["quality_code"].astype("category")

if len(calibration) > MAX_CALIBRATION_ROWS:
    raise MemoryError(
        f"Calibration contains {len(calibration):,} rows. "
        "Reduce ENTITY_LIMIT rather than sampling timestamps."
    )

calibration_duplicates = calibration.duplicated(
    ["event_ts", "entity_id", "metric_id"]
).sum()
print(f"Calibration rows: {len(calibration):,}")
print(f"Memory: {calibration.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Duplicate calibration keys: {calibration_duplicates:,}")


### Missingness map

The heatmap shows the proportion of valid rows in consecutive time blocks for
the entity with the most calibration data. It does not fill missing values.


In [ ]:
entity_counts = calibration["entity_id"].value_counts()
example_entity = str(entity_counts.index[0])
entity_data = calibration.loc[
    calibration["entity_id"].astype(str).eq(example_entity)
].copy()
entity_data["is_valid"] = entity_data["value"].notna().astype(float)

validity = entity_data.pivot_table(
    index="event_ts",
    columns="metric_id",
    values="is_valid",
    aggfunc="min",
).sort_index()

if len(validity) > 200:
    block = np.floor(
        np.arange(len(validity)) * 200 / len(validity)
    ).astype(int)
    validity = validity.groupby(block).mean()

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    validity.T,
    vmin=0,
    vmax=1,
    cmap="viridis",
    xticklabels=20,
    cbar_kws={"label": "Valid fraction"},
    ax=ax,
)
ax.set_title(f"Calibration validity map: {example_entity}")
ax.set_xlabel("Consecutive time blocks")
ax.set_ylabel("Metric")
save_and_show(fig, "missingness_map.png")


## 6. Robust distributions and entity heterogeneity

Statistics are calculated per entity and metric first. This prevents entities
with longer histories from silently dominating all conclusions.


In [ ]:
def median_absolute_deviation(values):
    values = np.asarray(values, dtype=float)
    if not len(values):
        return np.nan
    center = np.nanmedian(values)
    return float(np.nanmedian(np.abs(values - center)))


def block_level_scale_correlation(values, blocks=20):
    values = np.asarray(values, dtype=float)
    chunks = [
        chunk for chunk in np.array_split(values, blocks)
        if np.isfinite(chunk).sum() >= 5
    ]
    if len(chunks) < 4:
        return np.nan
    centers = [np.nanmedian(chunk) for chunk in chunks]
    scales = [median_absolute_deviation(chunk) for chunk in chunks]
    if np.nanstd(centers) == 0 or np.nanstd(scales) == 0:
        return np.nan
    return float(spearmanr(centers, scales).statistic)


def describe_series(group):
    group = group.sort_values("event_ts")
    values = group["value"].to_numpy(dtype=float)
    valid = values[np.isfinite(values)]
    if not len(valid):
        return {
            "rows": len(group), "n_valid": 0, "valid_rate": 0.0,
            "unique_values": 0,
        }
    mad = median_absolute_deviation(valid)
    first, second = np.array_split(valid, 2)
    drift = (
        (np.median(second) - np.median(first)) / mad
        if mad > 0 and len(first) and len(second) else np.nan
    )
    differences = np.diff(valid)
    return {
        "rows": len(group),
        "n_valid": len(valid),
        "valid_rate": len(valid) / len(group),
        "clipped_rate": group["quality_code"].eq("clipped").mean(),
        "unique_values": pd.Series(valid).nunique(),
        "zero_rate": np.mean(valid == 0),
        "minimum": np.min(valid),
        "p01": np.quantile(valid, 0.01),
        "p05": np.quantile(valid, 0.05),
        "p25": np.quantile(valid, 0.25),
        "median": np.median(valid),
        "p75": np.quantile(valid, 0.75),
        "p95": np.quantile(valid, 0.95),
        "p99": np.quantile(valid, 0.99),
        "maximum": np.max(valid),
        "mean": np.mean(valid),
        "std": np.std(valid, ddof=1) if len(valid) > 1 else 0.0,
        "mad": mad,
        "iqr": np.quantile(valid, 0.75) - np.quantile(valid, 0.25),
        "skewness": skew(valid, bias=False) if len(valid) > 2 else np.nan,
        "variance_mean_ratio": (
            np.var(valid, ddof=1) / np.mean(valid)
            if len(valid) > 1 and np.mean(valid) > 0 else np.nan
        ),
        "level_scale_correlation": block_level_scale_correlation(valid),
        "drift_effect_mad": drift,
        "negative_difference_rate": (
            np.mean(differences < 0) if len(differences) else np.nan
        ),
        "transition_rate": (
            np.mean(differences != 0) if len(differences) else np.nan
        ),
    }


In [ ]:
series_rows = []
for (entity_id, metric_id), group in calibration.groupby(
    ["entity_id", "metric_id"],
    observed=True,
    sort=True,
):
    row = {
        "entity_id": str(entity_id),
        "metric_id": str(metric_id),
        **describe_series(group),
    }
    series_rows.append(row)

series_profile = pd.DataFrame(series_rows)

pooled_rows = []
for metric_id, group in calibration.groupby(
    "metric_id", observed=True, sort=True
):
    summary = describe_series(group)
    entity_summary = series_profile.loc[
        series_profile["metric_id"].eq(str(metric_id))
    ]
    centers = entity_summary["median"].dropna().to_numpy()
    pooled_rows.append({
        "metric_id": str(metric_id),
        **summary,
        "series_count": len(entity_summary),
        "between_entity_mad": median_absolute_deviation(centers),
        "median_within_entity_mad": entity_summary["mad"].median(),
    })

metric_profile = (
    pd.DataFrame(pooled_rows)
    .merge(catalogue, on="metric_id", how="left")
    .merge(
        structural_profile[[
            "metric_id", "rows", "valid_rate",
            "invalid_rate", "clipped_rate", "zero_rate"
        ]].rename(columns={
            "rows": "full_rows",
            "valid_rate": "full_valid_rate",
            "invalid_rate": "full_invalid_rate",
            "clipped_rate": "full_clipped_rate",
            "zero_rate": "full_zero_rate",
        }),
        on="metric_id",
        how="left",
    )
)
display(metric_profile)


### Distribution plots

Every metric is plotted. Histograms use at most the central 99% for visual
resolution; the complete minimum, maximum and tail quantiles remain in the
profile table. Boxplots suppress individual fliers only in the drawing.


In [ ]:
metrics = catalogue["metric_id"].astype(str).tolist()
for page, start in enumerate(range(0, len(metrics), 5), start=1):
    page_metrics = metrics[start:start + 5]
    fig, axes = plt.subplots(
        len(page_metrics), 2,
        figsize=(13, 3.2 * len(page_metrics)),
        squeeze=False,
    )
    for row, metric_id in enumerate(page_metrics):
        values = calibration.loc[
            calibration["metric_id"].astype(str).eq(metric_id),
            "value",
        ].dropna()
        if len(values) > MAX_DISTRIBUTION_POINTS:
            values = values.sample(
                MAX_DISTRIBUTION_POINTS,
                random_state=42,
            )
        if values.empty:
            axes[row, 0].set_title(f"{metric_id}: no valid values")
            axes[row, 1].axis("off")
            continue
        lower, upper = values.quantile([0.005, 0.995])
        central = values.loc[values.between(lower, upper)]
        kind = catalogue.set_index("metric_id").loc[
            metric_id, "measurement_kind"
        ]
        sns.histplot(
            central,
            bins=40,
            kde=(kind == "gauge" and central.nunique() > 20),
            ax=axes[row, 0],
        )
        sns.boxplot(x=values, showfliers=False, ax=axes[row, 1])
        axes[row, 0].set_title(f"{metric_id}: central 99%")
        axes[row, 1].set_title(f"{metric_id}: boxplot")
    save_and_show(fig, f"distribution_page_{page:02d}.png")


## 7. Temporal behaviour, stationarity and autocorrelation

One representative entity is selected per metric using completeness and
sample size. ACF and PACF use the longest complete regular segment; missing
intervals are not interpolated.


In [ ]:
representatives = (
    series_profile.sort_values(
        ["metric_id", "valid_rate", "n_valid", "entity_id"],
        ascending=[True, False, False, True],
    )
    .groupby("metric_id", as_index=False)
    .first()[["metric_id", "entity_id", "valid_rate", "n_valid"]]
)

plot_metrics = []
for kind in catalogue["measurement_kind"].drop_duplicates():
    candidates = metric_profile.loc[
        metric_profile["measurement_kind"].eq(kind)
    ].sort_values("full_valid_rate", ascending=False)
    if len(candidates):
        plot_metrics.append(candidates.iloc[0]["metric_id"])
for metric_id in (
    metric_profile.sort_values("full_valid_rate", ascending=False)["metric_id"]
):
    if metric_id not in plot_metrics:
        plot_metrics.append(metric_id)
    if len(plot_metrics) >= MAX_TIME_PLOT_METRICS:
        break

expected_cadence = pd.Timedelta(
    seconds=float(core_manifest["expected_cadence_seconds"])
)


def regular_series(entity_id, metric_id):
    frame = calibration.loc[
        calibration["entity_id"].astype(str).eq(entity_id)
        & calibration["metric_id"].astype(str).eq(metric_id),
        ["event_ts", "value"],
    ].sort_values("event_ts")
    series = frame.drop_duplicates("event_ts").set_index("event_ts")["value"]
    return series.asfreq(expected_cadence)


def longest_complete_segment(series):
    valid = series.notna()
    if not valid.any():
        return series.iloc[:0]
    runs = valid.ne(valid.shift()).cumsum()
    valid_runs = runs.loc[valid]
    longest = valid_runs.value_counts().idxmax()
    return series.loc[runs.eq(longest)]


def stationarity_tests(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[:10_000]
    result = {
        "adf_statistic": np.nan,
        "adf_pvalue": np.nan,
        "kpss_statistic": np.nan,
        "kpss_pvalue": np.nan,
        "stationarity_evidence": "insufficient_data",
    }
    if len(values) < 100 or np.unique(values).size < 3:
        return result
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            adf_result = adfuller(values, maxlag=12, autolag="AIC")
            kpss_result = kpss(values, regression="c", nlags="auto")
    except (ValueError, np.linalg.LinAlgError):
        result["stationarity_evidence"] = "test_failed"
        return result
    result.update({
        "adf_statistic": adf_result[0],
        "adf_pvalue": adf_result[1],
        "kpss_statistic": kpss_result[0],
        "kpss_pvalue": kpss_result[1],
    })
    if adf_result[1] <= 0.05 and kpss_result[1] > 0.05:
        result["stationarity_evidence"] = "stationary_evidence"
    elif adf_result[1] > 0.05 and kpss_result[1] <= 0.05:
        result["stationarity_evidence"] = "nonstationary_evidence"
    else:
        result["stationarity_evidence"] = "mixed_evidence"
    return result


In [ ]:
temporal_rows = []
for row in representatives.itertuples(index=False):
    regular = regular_series(row.entity_id, row.metric_id)
    segment = longest_complete_segment(regular)
    values = segment.dropna()
    differences = values.diff().dropna()
    tests = (
        stationarity_tests(values)
        if row.metric_id in plot_metrics
        else {
            "adf_statistic": np.nan,
            "adf_pvalue": np.nan,
            "kpss_statistic": np.nan,
            "kpss_pvalue": np.nan,
            "stationarity_evidence": "not_tested_representative_limit",
        }
    )
    temporal_rows.append({
        "metric_id": row.metric_id,
        "entity_id": row.entity_id,
        "regular_rows": len(regular),
        "regular_missing_rate": regular.isna().mean(),
        "longest_complete_rows": len(values),
        "acf_1": values.autocorr(1) if len(values) > 2 else np.nan,
        "difference_acf_1": (
            differences.autocorr(1) if len(differences) > 2 else np.nan
        ),
        **tests,
    })

temporal_evidence = pd.DataFrame(temporal_rows)
display(temporal_evidence)


### Representative series with rolling robust statistics

The rolling window is 5% of the available regular calibration segment, bounded
between 12 and 500 observations. It is reported in both observations and
elapsed time.


In [ ]:
for metric_id in plot_metrics:
    entity_id = representatives.set_index("metric_id").loc[
        metric_id, "entity_id"
    ]
    series = regular_series(entity_id, metric_id)
    rolling_points = max(12, min(500, max(12, len(series) // 20)))
    rolling_median = series.rolling(
        rolling_points, min_periods=max(3, rolling_points // 3)
    ).median()
    rolling_q25 = series.rolling(
        rolling_points, min_periods=max(3, rolling_points // 3)
    ).quantile(0.25)
    rolling_q75 = series.rolling(
        rolling_points, min_periods=max(3, rolling_points // 3)
    ).quantile(0.75)

    step = max(1, len(series) // MAX_PLOT_POINTS)
    index = series.index[::step]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(index, series.iloc[::step], alpha=0.45, label="value")
    ax.plot(index, rolling_median.iloc[::step], color="black",
            label=f"rolling median ({rolling_points} points)")
    ax.fill_between(
        index,
        rolling_q25.iloc[::step],
        rolling_q75.iloc[::step],
        color="steelblue",
        alpha=0.20,
        label="rolling IQR",
    )
    duration = rolling_points * expected_cadence
    ax.set_title(f"{metric_id} | {entity_id} | window ≈ {duration}")
    ax.legend(loc="best")
    save_and_show(fig, f"series_{metric_id}.png")


### ACF and PACF

These plots are descriptive. They do not automatically select an AR model.


In [ ]:
for metric_id in plot_metrics:
    entity_id = representatives.set_index("metric_id").loc[
        metric_id, "entity_id"
    ]
    segment = longest_complete_segment(
        regular_series(entity_id, metric_id)
    ).dropna()
    values = segment.iloc[:10_000]
    if len(values) < 40 or values.nunique() < 3:
        print(f"Skipping ACF/PACF for {metric_id}: insufficient variation")
        continue
    lags = min(80, len(values) // 4)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    plot_acf(values, lags=lags, ax=axes[0], zero=False)
    plot_pacf(values, lags=lags, ax=axes[1], zero=False, method="ywm")
    axes[0].set_title(f"{metric_id}: ACF")
    axes[1].set_title(f"{metric_id}: PACF")
    save_and_show(fig, f"acf_pacf_{metric_id}.png")


## 8. Seasonality and robust STL

Daily and weekly calendar periods are tested only when the longest complete
segment contains at least six cycles. No gaps are interpolated. An
`insufficient_cycles` result is a valid finding, especially for short 3W event
traces.


In [ ]:
cadence_seconds = expected_cadence.total_seconds()
period_candidates = {
    "daily": round(86_400 / cadence_seconds),
    "weekly": round(604_800 / cadence_seconds),
}
period_candidates = {
    name: period
    for name, period in period_candidates.items()
    if period >= 2
}


def stl_strength(result):
    residual_variance = np.var(result.resid)
    trend_denominator = np.var(result.trend + result.resid)
    seasonal_denominator = np.var(result.seasonal + result.resid)
    return {
        "trend_strength": max(
            0.0, 1 - residual_variance / trend_denominator
        ) if trend_denominator > 0 else 0.0,
        "seasonal_strength": max(
            0.0, 1 - residual_variance / seasonal_denominator
        ) if seasonal_denominator > 0 else 0.0,
        "residual_scale": median_absolute_deviation(result.resid),
    }


seasonality_rows = []
stl_results = {}
for row in representatives.itertuples(index=False):
    segment = longest_complete_segment(
        regular_series(row.entity_id, row.metric_id)
    ).dropna()
    for period_name, period in period_candidates.items():
        cycles = len(segment) / period
        evidence = {
            "metric_id": row.metric_id,
            "entity_id": row.entity_id,
            "period_name": period_name,
            "period_observations": period,
            "cycles_available": cycles,
            "status": "insufficient_cycles",
            "trend_strength": np.nan,
            "seasonal_strength": np.nan,
            "residual_scale": np.nan,
        }
        if cycles >= MIN_STL_CYCLES and segment.nunique() >= 3:
            try:
                result = STL(
                    segment.astype(float),
                    period=period,
                    robust=True,
                ).fit()
            except (ValueError, np.linalg.LinAlgError):
                evidence["status"] = "fit_failed"
            else:
                evidence.update({
                    "status": "evaluated",
                    **stl_strength(result),
                })
                stl_results[(row.metric_id, period_name)] = result
        seasonality_rows.append(evidence)

seasonality_evidence = pd.DataFrame(seasonality_rows)
display(seasonality_evidence)


In [ ]:
evaluated = seasonality_evidence.loc[
    seasonality_evidence["status"].eq("evaluated")
].sort_values("seasonal_strength", ascending=False)

if evaluated.empty:
    print(
        "No series passed the STL gate. "
        "No decomposition was forced."
    )
else:
    for row in evaluated.head(6).itertuples(index=False):
        result = stl_results[(row.metric_id, row.period_name)]
        figure = result.plot()
        figure.set_size_inches(12, 8)
        figure.suptitle(
            f"{row.metric_id}: robust STL ({row.period_name})",
            y=1.01,
        )
        save_and_show(
            figure,
            f"stl_{row.metric_id}_{row.period_name}.png",
        )


## 9. Cross-metric dependence

Raw-level correlation can be driven by shared trend. We therefore report the
median within-entity Spearman correlation for both levels and first
differences. Discrete states are excluded from this continuous correlation
matrix and assessed through transition rates instead.


In [ ]:
continuous_metrics = (
    metric_profile.loc[
        ~metric_profile["measurement_kind"].eq("discrete_state")
    ]
    .sort_values("full_valid_rate", ascending=False)
    ["metric_id"]
    .head(MAX_CORRELATION_METRICS)
    .tolist()
)

level_matrices = []
difference_matrices = []
pair_count_matrices = []

for entity_id in selected_entities:
    entity = calibration.loc[
        calibration["entity_id"].astype(str).eq(entity_id)
        & calibration["metric_id"].astype(str).isin(continuous_metrics),
        ["event_ts", "metric_id", "value"],
    ]
    pivot = entity.pivot_table(
        index="event_ts",
        columns="metric_id",
        values="value",
        aggfunc="first",
    ).sort_index()
    pivot = pivot.reindex(columns=continuous_metrics)
    level_matrices.append(pivot.corr(method="spearman"))
    difference_matrices.append(pivot.diff().corr(method="spearman"))
    available = pivot.notna().astype(int)
    pair_count_matrices.append(available.T @ available)

level_array = np.stack([matrix.to_numpy() for matrix in level_matrices])
difference_array = np.stack([
    matrix.to_numpy() for matrix in difference_matrices
])
count_array = np.stack([
    matrix.to_numpy() for matrix in pair_count_matrices
])

median_level = pd.DataFrame(
    np.nanmedian(level_array, axis=0),
    index=continuous_metrics,
    columns=continuous_metrics,
)
median_difference = pd.DataFrame(
    np.nanmedian(difference_array, axis=0),
    index=continuous_metrics,
    columns=continuous_metrics,
)
median_pair_count = pd.DataFrame(
    np.nanmedian(count_array, axis=0),
    index=continuous_metrics,
    columns=continuous_metrics,
)

dependence_rows = []
for left_index, left in enumerate(continuous_metrics):
    for right_index in range(left_index + 1, len(continuous_metrics)):
        right = continuous_metrics[right_index]
        dependence_rows.append({
            "metric_left": left,
            "metric_right": right,
            "median_level_spearman": median_level.loc[left, right],
            "median_difference_spearman": median_difference.loc[left, right],
            "median_pair_observations": median_pair_count.loc[left, right],
            "entities_assessed": len(selected_entities),
        })
dependence_evidence = pd.DataFrame(dependence_rows)
display(
    dependence_evidence.reindex(
        dependence_evidence["median_difference_spearman"]
        .abs().sort_values(ascending=False).index
    ).head(20)
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(
    median_level,
    vmin=-1,
    vmax=1,
    center=0,
    cmap="vlag",
    ax=axes[0],
)
sns.heatmap(
    median_difference,
    vmin=-1,
    vmax=1,
    center=0,
    cmap="vlag",
    ax=axes[1],
)
axes[0].set_title("Median within-entity Spearman: levels")
axes[1].set_title("Median within-entity Spearman: differences")
save_and_show(fig, "correlation_levels_and_differences.png")


## 10. Statistical readiness for feature engineering

The final status concerns statistical usability only. It is not a statement
that a value is physically plausible or operationally important.


In [ ]:
best_seasonality = (
    seasonality_evidence.loc[
        seasonality_evidence["status"].eq("evaluated")
    ]
    .sort_values("seasonal_strength", ascending=False)
    .groupby("metric_id", as_index=False)
    .first()[[
        "metric_id", "period_name",
        "trend_strength", "seasonal_strength"
    ]]
)
metric_profile = metric_profile.merge(
    temporal_evidence,
    on="metric_id",
    how="left",
    suffixes=("", "_representative"),
).merge(
    best_seasonality,
    on="metric_id",
    how="left",
)


def candidate_representation(row):
    if row["measurement_kind"] == "cumulative_counter":
        return "counter_increment_with_reset_handling"
    if row["measurement_kind"] == "discrete_state":
        return "state_transitions_and_dwell"
    if row["measurement_kind"] == "interval_count":
        return "log1p_count" if row["skewness"] > 1 else "count_level"
    if row["measurement_kind"] == "bounded_fraction":
        return "bounded_level"
    positive = row["minimum"] >= 0
    multiplicative = (
        row["skewness"] > 1
        and row["level_scale_correlation"] > 0.5
    )
    return (
        "log1p_then_robust_center"
        if positive and multiplicative
        else "robust_centered_level"
    )


def readiness(row):
    if row["n_valid"] == 0 or row["unique_values"] <= 1:
        return "not_ready"
    if row["valid_rate"] < 0.50:
        return "limited"
    return "ready_for_baseline_research"


def caveat(row):
    notes = []
    if row["valid_rate"] < 0.90:
        notes.append("material_missingness")
    if row["full_clipped_rate"] > 0:
        notes.append("source_censoring")
    if abs(row["drift_effect_mad"]) > 2:
        notes.append("location_drift")
    if row["unique_values"] <= 2:
        notes.append("low_cardinality")
    return "; ".join(notes) or "none_identified_statistically"


metric_profile["candidate_representation"] = metric_profile.apply(
    candidate_representation, axis=1
)
metric_profile["data_readiness_status"] = metric_profile.apply(
    readiness, axis=1
)
metric_profile["caveat"] = metric_profile.apply(caveat, axis=1)

readiness_columns = [
    "metric_id", "measurement_kind", "unit",
    "full_valid_rate", "full_clipped_rate", "zero_rate",
    "series_count", "median_within_entity_mad", "between_entity_mad",
    "drift_effect_mad", "acf_1", "difference_acf_1",
    "stationarity_evidence", "period_name",
    "trend_strength", "seasonal_strength",
    "candidate_representation", "data_readiness_status", "caveat",
]
display(metric_profile[readiness_columns])


## 11. Save reproducible evidence

The output contains compact statistical evidence and figures—not a copied
telemetry dataset. The manifest records the canonical content hashes,
configuration, selected entities, and every file read.


In [ ]:
read_paths = sorted(
    str(path.relative_to(CORE_RESOLVED))
    for path in READ_FILES
)
assert all(
    (CORE_ROOT / relative).resolve().is_relative_to(CORE_RESOLVED)
    for relative in read_paths
)

output_tables = {
    "analysis_windows": analysis_windows,
    "metric_profile": metric_profile,
    "series_profile": series_profile,
    "temporal_evidence": temporal_evidence,
    "seasonality_evidence": seasonality_evidence,
    "dependence_evidence": dependence_evidence,
}
eda_manifest = {
    "eda_version": "0.1.0",
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "spec_core": str(CORE_ROOT),
    "spec_core_manifest_sha256": sha256_file(manifest_path),
    "canonical_content_hashes": core_manifest[
        "canonical_content_hashes"
    ],
    "configuration": {
        "calibration_fraction": CALIBRATION_FRACTION,
        "entity_limit": ENTITY_LIMIT,
        "max_calibration_rows": MAX_CALIBRATION_ROWS,
        "min_stl_cycles": MIN_STL_CYCLES,
    },
    "selected_entities": selected_entities,
    "files_read": read_paths,
    "truth_guard_passed": True,
    "duplicate_rows_within_partitions": duplicate_rows,
    "calibration_duplicate_keys": int(calibration_duplicates),
    "output_rows": {
        name: len(frame) for name, frame in output_tables.items()
    },
    "figures": sorted(path.name for path in FIGURE_CACHE.glob("*.png")),
    "scope_warning": (
        "The Petrobras 3W fixture tests the contract and is not a "
        "representative oil-well training population."
        if SECTOR == "petrobras_3w"
        else "Synthetic telecom evidence requires later real-data validation."
    ),
}

if SAVE_OUTPUTS:
    with immutable_directory(EDA_ROOT) as output:
        for name, frame in output_tables.items():
            frame.to_parquet(output / f"{name}.parquet", index=False)
        shutil.copytree(FIGURE_CACHE, output / "figures")
        write_json(output / "eda_manifest.json", eda_manifest)
    print("Saved EDA evidence:", EDA_ROOT)
else:
    print("SAVE_OUTPUTS=False: results were not written")

display(pd.Series({
    "truth_guard_passed": eda_manifest["truth_guard_passed"],
    "files_read": len(read_paths),
    "metrics_profiled": len(metric_profile),
    "series_profiled": len(series_profile),
    "figures": len(eda_manifest["figures"]),
    "next_stage": "feature engineering using frozen EDA evidence",
}, name="result").to_frame())

figure_temp.cleanup()
